# 2030 청년층의 미인지 대사이상 조기 선별 머신러닝 모델
**논문 프로젝트: 국민건강영양조사(KNHANES 2021-2024) 기반 분석**
- **연구 대상**: 만 19세 ~ 39세 청년층
- **목표변수(Target)**: 미인지 대사이상 고위험군(1) vs 정상군(0)
- **설명변수(Features)**: 비침습적(Non-invasive) 자가계측 및 생활습관 변수 (피를 뽑지 않는 설문형 변수)

In [ ]:
# 1. 필수 라이브러리 로드
import os
import warnings
import pandas as pd
import numpy as np
import pyreadstat
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')
print('필수 라이브러리가 성공적으로 로드되었습니다!')

In [ ]:
# 2. 2021~2024년 4개년도 데이터 로드 및 통합
data_dir = r'C:\knhanes'
years = ['21', '22', '23', '24']
df_list = []

# 분석에 사용할 핵심 컬럼 정의
selected_cols = [
    'ID', 'year', 'age', 'sex', 'incm', 'edu',
    # 기진단 및 약물 치료 여부 (제외 기준용)
    'DI1_dg', 'DI1_pt', 'DE1_dg', 'DE1_pt', 'DI2_dg', 'DI2_pt',
    # 실제 검진 수치 (Target 정의용)
    'HE_glu', 'HE_TG', 'HE_HDL_st2', 'HE_sbp', 'HE_dbp',
    # 비침습적 예측 변수 (Features)
    'HE_ht', 'HE_wt', 'HE_BMI', 'HE_wc',
    'sm_presnt', 'dr_month', 'pa_aerobic'
]

for y in years:
    fpath = os.path.join(data_dir, f'hn{y}_all.sas7bdat')
    if os.path.exists(fpath):
        print(f'20{y}년 데이터 읽는 중...')
        df_temp, _ = pyreadstat.read_sas7bdat(fpath)
        valid_cols = [c for c in selected_cols if c in df_temp.columns]
        df_list.append(df_temp[valid_cols])

df_all = pd.concat(df_list, axis=0, ignore_index=True)
print(f'\n통합 완료! 전체 데이터 크기: {df_all.shape}')
df_all.head()

In [ ]:
# 3. 2030 청년층(19~39세) 필터링 및 기진단자 제외
df_young = df_all[(df_all['age'] >= 19) & (df_all['age'] <= 39)].copy()
print(f'19~39세 청년층 수: {len(df_young):,}명')

# 기진단 약물 복용자(고혈압, 당뇨, 고지혈증 치료 중) 제외 (미인지 환자를 찾는 것이 목적)
med_mask = (df_young['DI1_pt'] == 1) | (df_young['DE1_pt'] == 1) | (df_young['DI2_pt'] == 1)
df_study = df_young[~med_mask].copy()
print(f'기진단 치료자 제외 후 최종 연구 대상자: {len(df_study):,}명')

In [ ]:
# 4. 목표변수(Target) 라벨링: 미인지 대사이상 고위험군(1) vs 정상(0)
# 검진 기준 (NCEP-ATP III / 당뇨학회):
# - 공복혈당 >= 100 mg/dL (전당뇨/당뇨)
# - 혈압 >= 130/85 mmHg
# - 중성지방 >= 150 mg/dL
# - HDL 콜레스테롤 < 40(남), < 50(여)

cond_glu = (df_study['HE_glu'] >= 100)
cond_bp = (df_study['HE_sbp'] >= 130) | (df_study['HE_dbp'] >= 85)
cond_tg = (df_study['HE_TG'] >= 150)
cond_hdl = ((df_study['sex'] == 1) & (df_study['HE_HDL_st2'] < 40)) | \
            ((df_study['sex'] == 2) & (df_study['HE_HDL_st2'] < 50))

is_abnormal = cond_glu | cond_bp | cond_tg | cond_hdl
df_study['Target'] = np.where(is_abnormal, 1, 0)

print('목표변수(Target) 분포:')
print(df_study['Target'].value_counts())
print(df_study['Target'].value_counts(normalize=True).round(4) * 100)

In [ ]:
# 5. 독립변수(Features) 준비 및 결측치 처리
# 파생변수 생성: 허리둘레-키 비율 (WHtR = Waist-to-Height Ratio, 최신 비만/대사 지표)
df_study['WHtR'] = df_study['HE_wc'] / df_study['HE_ht']

features = [
    'age', 'sex', 'incm', 'edu',
    'HE_BMI', 'HE_wc', 'WHtR',
    'sm_presnt', 'dr_month', 'pa_aerobic'
]

# 결측치가 있는 행 제거 (Clean Data)
data_clean = df_study.dropna(subset=features + ['Target']).copy()
print(f'결측치 제거 후 최종 분석 샘플 수: {len(data_clean):,}명')

X = data_clean[features]
y = data_clean['Target']

# Train / Test 분할 (7:3 stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print(f'Train 셋 크기: {X_train.shape}, Test 셋 크기: {X_test.shape}')

In [ ]:
# 6. 머신러닝 모델 학습 및 성능 평가 (Logistic, Random Forest, XGBoost, LightGBM)
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(eval_metric='logloss', random_state=42),
    'LightGBM': LGBMClassifier(random_state=42, verbose=-1)
}

results = []
plt.figure(figsize=(8, 6))

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    auc = roc_auc_score(y_test, y_pred_proba)
    pr_auc = average_precision_score(y_test, y_pred_proba)
    
    results.append({'Model': name, 'ROC-AUC': auc, 'PR-AUC': pr_auc})
    
    # ROC Curve 플롯
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random Chance')
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Sensitivity)')
plt.title('2030 Unaware Metabolic Risk Screening ROC Curves')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

result_df = pd.DataFrame(results).sort_values(by='ROC-AUC', ascending=False)
print('=== 모델별 성능 비교표 ===')
print(result_df.to_string(index=False))

In [ ]:
# 7. Feature Importance 시각화 (LightGBM 기준)
best_model = models['LightGBM']
importances = pd.Series(best_model.feature_importances_, index=features).sort_values(ascending=True)

plt.figure(figsize=(8, 5))
importances.plot(kind='barh', color='teal')
plt.title('Feature Importances for Unaware Metabolic Risk Screening (LightGBM)')
plt.xlabel('Importance')
plt.ylabel('Features')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()